# 智能手套ASL识别 - 集成学习方法
# Ensemble Learning Methods for ASL Recognition

## 项目概述

本Notebook探索多种集成学习方法，通过组合多个基学习器提高ASL手势识别性能。

### 集成学习核心思想

"三个臭皮匠，顶个诸葛亮" - 通过组合多个弱学习器构建强学习器

**核心公式**:
```
H(x) = Σ α_t · h_t(x)
```
其中 h_t(x) 是基学习器，α_t 是权重

### 涵盖方法
1. **Voting (投票法)** - 硬投票与软投票
2. **Stacking (堆叠法)** - 元学习器组合
3. **Blending (混合法)** - 保留集验证
4. **Bagging + Boosting组合** - 异构集成

### 数学基础

#### 1. 投票法 (Voting)

**硬投票 (Hard Voting)**:
```
ŷ = mode{h_1(x), h_2(x), ..., h_n(x)}
```
选择多数基学习器预测的类别

**软投票 (Soft Voting)**:
```
ŷ = argmax_{c} Σ w_i · P_i(y=c|x)
```
加权平均各基学习器的概率预测

#### 2. Stacking 堆叠法

**第一层 (基学习器)**:
```
h_1(x), h_2(x), ..., h_n(x)
```

**第二层 (元学习器)**:
```
H(x) = g(h_1(x), h_2(x), ..., h_n(x))
```
其中 g(·) 是元学习器（如Logistic Regression）

**预测过程**:
```
1. 用训练集训练基学习器 h_i
2. 用验证集生成预测: z_i = h_i(x)
3. 用 (z_1, ..., z_n) 训练元学习器 g
4. 最终预测: H(x) = g(h_1(x), ..., h_n(x))
```

#### 3. 加权平均

**权重学习**:
```
min_{w} Σ (y_i - Σ w_j · h_j(x_i))²
s.t. Σ w_j = 1, w_j ≥ 0
```

**优化方法**:
- 网格搜索
- 梯度下降
- 遗传算法

## 第一步：环境配置

In [ ]:
# 基础库
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.optimize import minimize
import warnings
warnings.filterwarnings('ignore')

# scikit-learn
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, classification_report

# 基学习器
from sklearn.ensemble import (RandomForestClassifier, VotingClassifier, 
                             StackingClassifier, BaggingClassifier, GradientBoostingClassifier)
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB

# XGBoost和LightGBM
import xgboost as xgb
import lightgbm as lgb

# 设置中文
plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

print("✅ 库导入成功！")

## 第二步：数据加载

In [ ]:
# 加载数据（复用之前的数据加载代码）
GESTURE_CLASSES = [
    'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm',
    'bad', 'deaf', 'fine', 'good', 'goodbye', 'hello', 
    'hungry', 'me', 'no', 'please'
]

def load_and_preprocess(data_path, gesture_classes):
    """加载并预处理数据"""
    all_data = []
    
    for gesture in gesture_classes:
        file_path = os.path.join(data_path, f'{gesture}_merged.csv_exported.csv')
        if os.path.exists(file_path):
            df = pd.read_csv(file_path)
            feature_cols = ['flex_1', 'flex_2', 'flex_3', 'flex_4', 'flex_5',
                          'GYRx', 'GYRy', 'GYRz', 'ACCx', 'ACCy', 'ACCz']
            df = df[['Alphabet'] + feature_cols].dropna()
            df[feature_cols[:5]] = df[feature_cols[:5]].clip(lower=0)
            all_data.append(df)
    
    combined_df = pd.concat(all_data, ignore_index=True)
    
    # 特征和标签
    feature_cols = ['flex_1', 'flex_2', 'flex_3', 'flex_4', 'flex_5',
                   'GYRx', 'GYRy', 'GYRz', 'ACCx', 'ACCy', 'ACCz']
    X = combined_df[feature_cols].values
    y = combined_df['Alphabet'].values
    
    # 归一化
    flex_max = X[:, :5].max()
    X[:, :5] = X[:, :5] / flex_max if flex_max > 0 else X[:, :5]
    X[:, 5:8] = X[:, 5:8] / 9.81
    gyro_max = np.abs(X[:, 8:11]).max()
    X[:, 8:11] = X[:, 8:11] / gyro_max if gyro_max > 0 else X[:, 8:11]
    
    # 编码
    le = LabelEncoder()
    y_encoded = le.fit_transform(y)
    
    return X, y_encoded, le

# 加载
X, y, label_encoder = load_and_preprocess('modified dataset/alphabet/', GESTURE_CLASSES)

# 划分数据
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

# 进一步划分训练集为训练和验证（用于Stacking）
X_train_base, X_val, y_train_base, y_val = train_test_split(
    X_train, y_train, test_size=0.2, random_state=42, stratify=y_train
)

print(f"训练集 (基学习器): {X_train_base.shape[0]}")
print(f"验证集 (元学习器): {X_val.shape[0]}")
print(f"测试集 (最终评估): {X_test.shape[0]}")
print(f"类别数: {len(label_encoder.classes_)}")

## 第三步：定义基学习器

In [ ]:
# 定义基学习器
base_learners = {
    'rf': RandomForestClassifier(
        n_estimators=200, max_depth=10, random_state=42, n_jobs=-1
    ),
    'svm': SVC(C=10, kernel='rbf', probability=True, random_state=42),
    'knn': KNeighborsClassifier(n_neighbors=5, weights='distance', n_jobs=-1),
    'xgb': xgb.XGBClassifier(
        n_estimators=200, max_depth=6, learning_rate=0.1,
        random_state=42, n_jobs=-1, verbosity=0
    ),
    'lgb': lgb.LGBMClassifier(
        n_estimators=200, num_leaves=31, learning_rate=0.1,
        random_state=42, n_jobs=-1, verbose=-1
    ),
    'dt': DecisionTreeClassifier(max_depth=10, random_state=42),
    'nb': GaussianNB()
}

print("定义的基学习器:")
for name, model in base_learners.items():
    print(f"  - {name}: {model.__class__.__name__}")

## 第四步：Voting (投票法)

In [ ]:
# 硬投票 (Hard Voting)
print("\n" + "="*80)
print("🗳️ 硬投票 (Hard Voting)")
print("="*80)

# 选择部分基学习器进行投票
voting_estimators = [
    ('rf', base_learners['rf']),
    ('xgb', base_learners['xgb']),
    ('lgb', base_learners['lgb']),
    ('svm', base_learners['svm'])
]

hard_voting = VotingClassifier(
    estimators=voting_estimators,
    voting='hard',  # 硬投票
    n_jobs=-1
)

hard_voting.fit(X_train, y_train)
hard_pred = hard_voting.predict(X_test)
hard_acc = accuracy_score(y_test, hard_pred)

print(f"硬投票准确率: {hard_acc:.4f}")

# 查看各基学习器性能
print("\n基学习器性能:")
for name, estimator in voting_estimators:
    acc = estimator.fit(X_train, y_train).score(X_test, y_test)
    print(f"  {name}: {acc:.4f}")

In [ ]:
# 软投票 (Soft Voting)
print("\n" + "="*80)
print("🗳️ 软投票 (Soft Voting)")
print("="*80)

soft_voting = VotingClassifier(
    estimators=voting_estimators,
    voting='soft',  # 软投票
    n_jobs=-1
)

soft_voting.fit(X_train, y_train)
soft_pred = soft_voting.predict(X_test)
soft_acc = accuracy_score(y_test, soft_pred)

print(f"软投票准确率: {soft_acc:.4f}")

# 软投票原理演示
print("\n软投票原理 (示例):")
sample_idx = 0
probas = []
for name, estimator in voting_estimators:
    proba = estimator.predict_proba(X_test[sample_idx:sample_idx+1])[0]
    probas.append(proba)
    predicted_class = np.argmax(proba)
    print(f"  {name}: 类别{predicted_class} (概率{proba[predicted_class]:.3f})")

# 平均概率
avg_proba = np.mean(probas, axis=0)
final_pred = np.argmax(avg_proba)
print(f"\n平均概率: 类别{final_pred} (概率{avg_proba[final_pred]:.3f})")
print(f"真实标签: 类别{y_test[sample_idx]}")

In [ ]:
# 加权投票
print("\n" + "="*80)
print("⚖️ 加权投票 (Weighted Voting)")
print("="*80)

# 使用不同权重
weights_list = [
    [1, 1, 1, 1],      # 等权重
    [2, 1, 1, 1],      # RF权重更高
    [1, 2, 1, 1],      # XGB权重更高
    [1, 1, 2, 1],      # LGB权重更高
    [1, 1, 1, 2],      # SVM权重更高
    [2, 2, 1, 1],      # RF+XGB
]

weight_results = []
for weights in weights_list:
    weighted_voting = VotingClassifier(
        estimators=voting_estimators,
        voting='soft',
        weights=weights,
        n_jobs=-1
    )
    weighted_voting.fit(X_train, y_train)
    acc = weighted_voting.score(X_test, y_test)
    weight_results.append({'weights': weights, 'accuracy': acc})
    print(f"权重 {weights}: 准确率 {acc:.4f}")

# 找出最佳权重
best_weight = max(weight_results, key=lambda x: x['accuracy'])
print(f"\n最佳权重: {best_weight['weights']}, 准确率: {best_weight['accuracy']:.4f}")

## 第五步：Stacking (堆叠法)

In [ ]:
# 手动实现 Stacking
print("\n" + "="*80)
print("📚 Stacking (堆叠法) - 手动实现")
print("="*80)

# Step 1: 训练基学习器
print("\nStep 1: 训练基学习器...")
base_predictions_train = {}
base_predictions_val = {}

for name, model in base_learners.items():
    print(f"  训练 {name}...")
    model.fit(X_train_base, y_train_base)
    base_predictions_train[name] = model.predict_proba(X_train_base)
    base_predictions_val[name] = model.predict_proba(X_val)

# Step 2: 构建元特征
print("\nStep 2: 构建元特征...")
num_classes = len(label_encoder.classes_)

# 将各基学习器的概率预测拼接为元特征
meta_features_train = np.hstack([
    base_predictions_train[name] for name in base_learners.keys()
])

meta_features_val = np.hstack([
    base_predictions_val[name] for name in base_learners.keys()
])

print(f"  元特征维度: {meta_features_train.shape}")

# Step 3: 训练元学习器
print("\nStep 3: 训练元学习器...")
meta_learner = LogisticRegression(max_iter=1000, random_state=42)
meta_learner.fit(meta_features_train, y_train_base)

# Step 4: 验证集评估
meta_pred_val = meta_learner.predict(meta_features_val)
meta_acc_val = accuracy_score(y_val, meta_pred_val)
print(f"  验证集准确率: {meta_acc_val:.4f}")

In [ ]:
# Stacking 测试集预测
print("\nStep 4: 测试集预测...")

# 基学习器对测试集预测
base_predictions_test = {}
for name, model in base_learners.items():
    base_predictions_test[name] = model.predict_proba(X_test)

# 构建测试集元特征
meta_features_test = np.hstack([
    base_predictions_test[name] for name in base_learners.keys()
])

# 元学习器预测
stacking_pred = meta_learner.predict(meta_features_test)
stacking_acc = accuracy_score(y_test, stacking_pred)

print(f"\nStacking 测试集准确率: {stacking_acc:.4f}")

# 对比基学习器
print("\n基学习器对比:")
for name, model in base_learners.items():
    pred = model.predict(X_test)
    acc = accuracy_score(y_test, pred)
    print(f"  {name:10s}: {acc:.4f}")

In [ ]:
# 使用 sklearn 的 StackingClassifier
print("\n" + "="*80)
print("📚 Stacking - sklearn实现")
print("="*80)

stacking_sklearn = StackingClassifier(
    estimators=[
        ('rf', base_learners['rf']),
        ('xgb', base_learners['xgb']),
        ('lgb', base_learners['lgb']),
        ('svm', base_learners['svm']),
        ('knn', base_learners['knn'])
    ],
    final_estimator=LogisticRegression(max_iter=1000),
    cv=5,  # 5折交叉验证
    stack_method='predict_proba',
    n_jobs=-1
)

stacking_sklearn.fit(X_train, y_train)
stacking_sklearn_acc = stacking_sklearn.score(X_test, y_test)

print(f"Stacking (sklearn) 准确率: {stacking_sklearn_acc:.4f}")

## 第六步：Blending (混合法)

In [ ]:
# Blending 实现
print("\n" + "="*80)
print("🥤 Blending (混合法)")
print("="*80)

# Blending 与 Stacking 的区别:
# - Stacking 使用交叉验证生成元特征
# - Blending 使用保留集(hold-out set)生成元特征

# 使用 X_val, y_val 作为保留集（已在数据划分时准备好）

# Step 1: 在训练集上训练基学习器
print("\nStep 1: 训练基学习器...")
blend_models = {}
for name, model in base_learners.items():
    model.fit(X_train_base, y_train_base)
    blend_models[name] = model
    print(f"  {name} 完成")

# Step 2: 在保留集上预测，构建混合特征
print("\nStep 2: 构建混合特征...")
blend_features = []
for name, model in blend_models.items():
    pred = model.predict_proba(X_val)
    blend_features.append(pred)

blend_features = np.hstack(blend_features)
print(f"  混合特征维度: {blend_features.shape}")

# Step 3: 训练混合器
print("\nStep 3: 训练混合器...")
blender = LogisticRegression(max_iter=1000, random_state=42)
blender.fit(blend_features, y_val)

# Step 4: 测试集预测
print("\nStep 4: 测试集预测...")
test_blend_features = []
for name, model in blend_models.items():
    pred = model.predict_proba(X_test)
    test_blend_features.append(pred)

test_blend_features = np.hstack(test_blend_features)
blend_pred = blender.predict(test_blend_features)
blend_acc = accuracy_score(y_test, blend_pred)

print(f"\nBlending 准确率: {blend_acc:.4f}")

## 第七步：权重优化

In [ ]:
# 基于验证集的权重优化
print("\n" + "="*80)
print("🔍 权重优化")
print("="*80)

# 获取基学习器在验证集上的预测概率
val_predictions = []
for name, model in base_learners.items():
    model.fit(X_train_base, y_train_base)
    pred_proba = model.predict_proba(X_val)
    val_predictions.append(pred_proba)

val_predictions = np.array(val_predictions)  # (n_models, n_samples, n_classes)

# 定义目标函数：负准确率（最小化）
def objective(weights):
    """
    优化目标：最小化负准确率
    
    加权预测: y_pred = argmax(Σ w_i * P_i)
    """
    weights = np.array(weights)
    weights = weights / weights.sum()  # 归一化
    
    # 加权平均概率
    weighted_proba = np.tensordot(weights, val_predictions, axes=([0], [0]))
    y_pred = np.argmax(weighted_proba, axis=1)
    
    # 返回负准确率（因为要最小化）
    return -accuracy_score(y_val, y_pred)

# 使用scipy优化
from scipy.optimize import minimize

n_models = len(base_learners)
initial_weights = np.ones(n_models) / n_models
bounds = [(0, 1) for _ in range(n_models)]
constraints = {'type': 'eq', 'fun': lambda w: np.sum(w) - 1}

result = minimize(
    objective, 
    initial_weights, 
    method='SLSQP',
    bounds=bounds,
    constraints=constraints
)

optimal_weights = result.x
optimal_weights = optimal_weights / optimal_weights.sum()

print("\n最优权重:")
for name, weight in zip(base_learners.keys(), optimal_weights):
    print(f"  {name}: {weight:.4f}")

# 使用最优权重在测试集上评估
test_predictions = []
for name, model in base_learners.items():
    pred_proba = model.predict_proba(X_test)
    test_predictions.append(pred_proba)

test_predictions = np.array(test_predictions)
weighted_test_proba = np.tensordot(optimal_weights, test_predictions, axes=([0], [0]))
weighted_pred = np.argmax(weighted_test_proba, axis=1)
weighted_acc = accuracy_score(y_test, weighted_pred)

print(f"\n最优加权准确率: {weighted_acc:.4f}")

## 第八步：结果对比

In [ ]:
# 汇总所有结果
results = {
    'Hard Voting': hard_acc,
    'Soft Voting': soft_acc,
    f"Weighted\n({str(best_weight['weights'])})": best_weight['accuracy'],
    'Stacking (Manual)': stacking_acc,
    'Stacking (sklearn)': stacking_sklearn_acc,
    'Blending': blend_acc,
    'Optimal Weighted': weighted_acc
}

# 添加最佳基学习器作为对比
best_base_acc = 0
best_base_name = ''
for name, model in base_learners.items():
    model.fit(X_train, y_train)
    acc = model.score(X_test, y_test)
    if acc > best_base_acc:
        best_base_acc = acc
        best_base_name = name

results[f'Best Base\n({best_base_name})'] = best_base_acc

# 可视化
plt.figure(figsize=(14, 8))
methods = list(results.keys())
accuracies = list(results.values())

# 使用不同颜色区分集成方法和基学习器
colors = ['#2ecc71' if 'Base' not in method else '#e74c3c' for method in methods]

bars = plt.bar(methods, accuracies, color=colors, alpha=0.7, edgecolor='black')
plt.ylabel('准确率')
plt.title('集成学习方法对比', fontsize=16)
plt.xticks(rotation=45, ha='right')
plt.ylim(0.8, 1.0)
plt.grid(axis='y', alpha=0.3)

# 添加数值标签
for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height,
            f'{height:.4f}', ha='center', va='bottom')

# 添加图例
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#2ecc71', alpha=0.7, label='集成方法'),
    Patch(facecolor='#e74c3c', alpha=0.7, label='最佳基学习器')
]
plt.legend(handles=legend_elements)

plt.tight_layout()
plt.savefig('ensemble_comparison.png', dpi=200)
plt.show()

# 打印结果表
print("\n" + "="*80)
print("📊 最终对比结果")
print("="*80)
for method, acc in sorted(results.items(), key=lambda x: x[1], reverse=True):
    print(f"{method:25s}: {acc:.4f}")

## 第九步：总结与建议

In [ ]:
print("\n" + "="*80)
print("📝 集成学习总结")
print("="*80)

print("""
集成学习核心思想:
  通过组合多个学习器，利用集体智慧提高预测性能

方法对比:

1. Voting (投票法)
   - 硬投票: 简单多数表决
   - 软投票: 概率加权平均
   - 优点: 实现简单，无需重新训练
   - 缺点: 基学习器必须差异大才能提升

2. Stacking (堆叠法)
   - 使用元学习器组合基学习器输出
   - 优点: 灵活性高，通常效果最好
   - 缺点: 容易过拟合，需要仔细设计

3. Blending (混合法)
   - 使用保留集而不是交叉验证
   - 优点: 简单快速
   - 缺点: 数据利用不充分

4. 权重优化
   - 基于验证集学习最优权重
   - 优点: 自适应权重
   - 缺点: 优化计算开销

部署建议:
  - 资源充足: 使用Stacking，效果最佳
  - 资源有限: 使用Voting，简单高效
  - 实时性要求高: 使用轻量级基学习器 + 缓存
""")

# 找出最佳方法
best_method = max(results.items(), key=lambda x: x[1])
print(f"\n🏆 最佳方法: {best_method[0]}")
print(f"   准确率: {best_method[1]:.4f}")
improvement = (best_method[1] - best_base_acc) / best_base_acc * 100
print(f"   相比最佳基学习器提升: {improvement:.2f}%")